# Lab 7: Training bigger networks

Two things can go wrong when training a network: it can fit the training data but fail on new data (**overfitting**), or it can fail to fit the training data at all (**optimization failure**). This lab studies each in turn, on handwritten letters, together with the tools that fix them. Each part follows the same path: observe the problem on a baseline run, then implement the tools that fix it and measure their effect. Part 3 is an optional stretch goal.

Exercises that ask you to write code in `minitorch` come with the terminal command that tests them. Run it from the root of the repository.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

import numpy as np
import matplotlib.pyplot as plt

from minitorch import init
from minitorch.layer import Linear, Sequential, ReLU, Sigmoid, BatchNorm1d, Residual
from minitorch.loss import CrossEntropyLoss
from minitorch.optim import SGD
from minitorch.data import DataLoader, load_emnist
from minitorch.transforms import Compose, Normalize, RandomCrop, Flatten

np.random.seed(0)

## Part 0: Data and provided code

This part is provided: you only need to know what the functions do, as described in the text. Reading the code is optional. It is a good way to review backpropagation and training loops, but only if you have time left once the exercises are done: the main goal of this lab is the `minitorch` code you write.

### 0.1 EMNIST letters

`load_emnist` returns images of handwritten letters, as arrays of shape $(N, 28, 28)$ with pixels in $[0, 255]$, and labels from 0 (a) to 25 (z). The last 2 000 images are held out for validation.

In [ ]:
(x_train, y_train), _ = load_emnist()

x_pool, y_pool = x_train[:-2000], y_train[:-2000]   # training samples are drawn from here
x_val, y_val = x_train[-2000:], y_train[-2000:]     # validation set
N_CLASSES = 26

print(x_pool.shape, x_val.shape, x_pool.min(), x_pool.max())

fig, axes = plt.subplots(2, 13, figsize=(13, 2.4))
for letter, ax in enumerate(axes.flat):
    ax.imshow(x_pool[y_pool == letter][0], cmap="gray")
    ax.set_title("abcdefghijklmnopqrstuvwxyz"[letter])
    ax.axis("off")
plt.show()

### 0.2 Training

- `make_loaders` builds the data loaders: images are scaled to $[0, 1]$ and flattened, a fraction `label_noise` of the training labels can be replaced by a wrong class, and random crops can be applied to the training images. Corrupting labels is something you would never do on purpose in a real project: it is only a lab device, to make overfitting much more visible. Random crops, on the other hand, are a widespread data augmentation technique.
- `make_mlp` builds a fully connected network of `depth` hidden layers of `width` units.
- `fit` trains a network and returns `history`: a dictionary with four lists, `"train_loss"`, `"val_loss"`, `"train_acc"` and `"val_acc"`, with one value per epoch. It puts the network in **training mode** (`model.train()`) for the updates, and `evaluate` puts it in **eval mode** (`model.eval()`) to measure the metrics: some layers, such as batch normalization, behave differently in the two modes.
- `layer_grad_norms` runs one backward pass and returns the norm of the gradient reaching the input of each layer, first layer first: the learning signal available to all the layers before it.

In [ ]:
def corrupt_labels(y, fraction, seed=0):
    """Replace a fraction of the labels by a different class chosen at random (always the same ones)."""
    rng = np.random.RandomState(seed)
    y = y.copy()
    wrong = rng.choice(len(y), int(fraction * len(y)), replace=False)
    y[wrong] = (y[wrong] + rng.randint(1, N_CLASSES, len(wrong))) % N_CLASSES
    return y


def make_loaders(n_train, batch_size, label_noise=0.0, crop_padding=0):
    x, y = x_pool[:n_train], corrupt_labels(y_pool[:n_train], label_noise)
    prepare = [Normalize(0.0, 255.0), Flatten()]
    augment = [RandomCrop(28, padding=crop_padding)] if crop_padding else []
    train_loader = DataLoader(x, y, batch_size=batch_size, shuffle=True, transform=Compose(augment + prepare))
    clean_loader = DataLoader(x, y, batch_size=500, transform=Compose(prepare))   # training set, no augmentation
    val_loader = DataLoader(x_val, y_val, batch_size=500, transform=Compose(prepare))
    return train_loader, clean_loader, val_loader


def make_mlp(depth, width, activation=ReLU, norm=False, residual=False, in_features=784, out_features=N_CLASSES):
    def block(n_in, n_out):
        layers = [Linear(n_in, n_out)]
        if norm:
            layers.append(BatchNorm1d(n_out))
        layers.append(activation())
        return Sequential(*layers)

    layers = [block(in_features, width)]
    for _ in range(depth - 1):
        b = block(width, width)
        layers.append(Residual(b) if residual else b)
    layers.append(Linear(width, out_features))
    return Sequential(*layers)


def evaluate(model, loss_fn, loader):
    """Mean loss and accuracy of the model over a loader, in eval mode."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        out = model(x)
        total_loss += loss_fn(out, y) * len(x)
        correct += np.sum(np.argmax(out, axis=1) == y)
        n += len(x)
    return total_loss / n, correct / n


def fit(model, loss_fn, optimizer, train_loader, clean_train_loader, val_loader, epochs, early_stopping=None):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            model.backward(loss_fn.backward())
            optimizer.step()
        for name, loader in (("train", clean_train_loader), ("val", val_loader)):
            loss, acc = evaluate(model, loss_fn, loader)
            history[name + "_loss"].append(loss)
            history[name + "_acc"].append(acc)
        if early_stopping is not None and early_stopping.step(history["val_loss"][-1], model):
            break
    if early_stopping is not None:
        early_stopping.restore(model)
    return history


def layer_grad_norms(model, loss_fn, x, y):
    """Norm of the gradient of the loss with respect to the input of each layer of model (first layer first)."""
    model.train()
    model.zero_grad()
    loss_fn(model(x), y)
    grad = loss_fn.backward()
    norms = []
    for layer in reversed(model.layers):      # same loop as Sequential.backward
        grad = layer.backward(grad)
        norms.append(np.linalg.norm(grad))
    model.zero_grad()
    return norms[::-1]

### 0.3 Experiments

Two networks are studied. Both are trained with SGD on the same 1 024 letters, 20% of which have a wrong label, from weights drawn from $\mathcal{N}(0, 0.05^2)$:

- the **wide** network (2 hidden layers of 256 units) is used to study overfitting;
- the **deep** network (10 hidden layers of 64 units) is used to study optimization failures: this small initialization is harmless for the wide network, but not for the deep one.

The **baseline** of a network is its training without any tool.

`run_and_record(name, net, **changes)` trains the network `net` from its configuration in `CONFIG`, overridden by `changes`, and stores the result in `results[(name, net)]`. `plot_runs` plots the loss (left) and accuracy (right) of experiments. `compare_tool` plots a tool against the baseline. `show_results` prints the final metrics.

In [ ]:
CONFIG = {
    "wide": dict(n_train=1024, label_noise=0.2, depth=2, width=256, init=0.05, lr=0.05, epochs=500, batch_size=128),
    "deep": dict(n_train=1024, label_noise=0.2, depth=10, width=64, init=0.05, lr=0.05, epochs=500, batch_size=128),
}
DEFAULTS = dict(norm=False, residual=False, crop_padding=0, weight_decay=0.0, patience=None)


def init_weights(model, init_fn):
    """Apply init_fn to every weight matrix of the model (biases and BatchNorm parameters keep their values)."""
    for p in model.parameters():
        if p.ndim == 2:
            init_fn(p)


def run_experiment(net, **changes):
    cfg = {**CONFIG[net], **DEFAULTS, **changes}
    np.random.seed(0)
    train_loader, clean_loader, val_loader = make_loaders(cfg["n_train"], cfg["batch_size"],
                                                          cfg["label_noise"], cfg["crop_padding"])
    model = make_mlp(cfg["depth"], cfg["width"], norm=cfg["norm"], residual=cfg["residual"])
    if cfg["init"] == "he":
        init_weights(model, init.kaiming_normal_)
    else:
        init_weights(model, lambda p: init.normal_(p, std=cfg["init"]))

    loss_fn = CrossEntropyLoss()
    xb, yb = next(iter(train_loader))
    norms = layer_grad_norms(model, loss_fn, xb, yb)              # at initialization

    optimizer = SGD(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    stopper = EarlyStopping(cfg["patience"]) if cfg["patience"] else None
    history = fit(model, loss_fn, optimizer, train_loader, clean_loader, val_loader, cfg["epochs"],
                  early_stopping=stopper)

    train_loss, train_acc = evaluate(model, loss_fn, clean_loader)    # final metrics, with the best weights
    val_loss, val_acc = evaluate(model, loss_fn, val_loader)          # when early stopping is used
    return dict(history=history, epochs_run=len(history["val_loss"]),
                train_loss=train_loss, val_loss=val_loss, train_acc=train_acc, val_acc=val_acc,
                grad_norms=norms, grad_ratio=norms[0] / norms[-1])


results = {}


def describe(metrics):
    """One line with the final train/val losses and train/val accuracies."""
    last = {k: (v[-1] if isinstance(v, list) else v) for k, v in metrics.items()}
    return (f"loss: train {last['train_loss']:.3f} / val {last['val_loss']:.3f} | "
            f"acc: train {last['train_acc']:.3f} / val {last['val_acc']:.3f}")


def run_and_record(name, net, **changes):
    results[(name, net)] = run_experiment(net, **changes)
    r = results[(name, net)]
    print(f"{name} [{net}] {describe(r)} | {r['epochs_run']} epochs")


TRAIN, VAL = "tab:blue", "tab:orange"             # the same colors in every plot


def draw_run(ax_loss, ax_acc, net, name, alpha=1.0):
    """Draw the loss and accuracy curves of one experiment: training in blue, validation in orange."""
    history = results[(name, net)]["history"]
    for key, color in (("train", TRAIN), ("val", VAL)):
        ax_loss.plot(history[key + "_loss"], color=color, alpha=alpha, label=f"{name} ({key})")
        ax_acc.plot(history[key + "_acc"], color=color, alpha=alpha, label=f"{name} ({key})")


def plot_runs(names, net):
    """One row per experiment of `names` on the network `net`: loss on the left, accuracy on the right."""
    fig, axes = plt.subplots(len(names), 2, figsize=(11, 3.2 * len(names)), squeeze=False)
    for row, name in enumerate(names):
        draw_run(axes[row, 0], axes[row, 1], net, name)
        axes[row, 1].set_ylim(0, 1.02)
        axes[row, 0].set_title(f"{name} [{net}]: loss")
        axes[row, 1].set_title(f"{name} [{net}]: accuracy")
        for ax in axes[row]:
            ax.set_xlabel("epoch")
        axes[row, 0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def compare_tool(tool, nets=("wide", "deep")):
    """The baseline (faded) and `tool` on the networks `nets`: loss on the left, accuracy on the right."""
    nets = [net for net in nets if (tool, net) in results]
    fig, axes = plt.subplots(len(nets), 2, figsize=(13, 4.2 * len(nets)), squeeze=False)
    for row, net in enumerate(nets):
        draw_run(axes[row, 0], axes[row, 1], net, "baseline", alpha=0.35)
        draw_run(axes[row, 0], axes[row, 1], net, tool)
        axes[row, 1].set_ylim(0, 1.02)
        axes[row, 0].set_title(f"[{net}] loss")
        axes[row, 1].set_title(f"[{net}] accuracy")
        for ax in axes[row]:
            ax.set_xlabel("epoch")
        axes[row, 0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def show_results(net, names):
    """Final metrics of the experiments `names` on the network `net`."""
    print(f"[{net}]{'':10s}| {'train loss':>10s} {'val loss':>9s} | {'train acc':>9s} {'val acc':>8s} | {'grad_ratio':>10s}")
    for name in names:
        if (name, net) in results:
            r = results[(name, net)]
            print(f"{name:16s}| {r['train_loss']:10.3f} {r['val_loss']:9.3f} | {r['train_acc']:9.3f} "
                  f"{r['val_acc']:8.3f} | {r['grad_ratio']:10.1e}")

## Part 1: Overfitting

The **wide** network has about 270 000 parameters for only 1 024 training letters, 20% of which have a wrong label (see Part 0.2).

In [ ]:
run_and_record("baseline", "wide")
plot_runs(["baseline"], "wide")

**Question 1.1.** The training accuracy ends up high although 20% of the labels are wrong. What does this tell you about what the network has learned? Which curves show that something goes wrong, and from which epoch?

*Answer:*

We test three tools that act on different levers: the data (random crops), the weights (weight decay) and the training duration (early stopping).

### Exercise 1.1: Random crops

A transform is a callable that the `DataLoader` applies to every batch of images, of shape $(N, 28, 28)$. Implement `RandomCrop(size, padding)` in `minitorch/transforms.py`: it pads every image with `padding` zeros on each side, then crops a `size` $\times$ `size` window at a random position, chosen independently for each image. With `size=28` and `padding=2`, every image is shifted by up to 2 pixels. Use `np.random`.

Test it with `python -m pytest tests/test_transforms.py`.

In [ ]:
crop = RandomCrop(28, padding=2)
batch = np.repeat(x_pool[:1], 8, axis=0)          # eight copies of the same letter

fig, axes = plt.subplots(1, 8, figsize=(12, 1.5))
for ax, img in zip(axes, crop(batch)):
    ax.imshow(img, cmap="gray")
    ax.axis("off")
plt.show()

run_and_record("random crop", "wide", crop_padding=2)
compare_tool("random crop", ["wide"])

### Exercise 1.2: Weight decay

Large weights let a network fit every label, wrong ones included; penalizing them favors smoother functions. Weight decay minimizes $L(\boldsymbol{\theta}) + \frac{\lambda}{2}\|\boldsymbol{\theta}\|^2$, where $\boldsymbol{\theta}$ gathers all the parameters; the gradient of this objective is $\nabla L(\boldsymbol{\theta}) + \lambda\,\boldsymbol{\theta}$.

In `minitorch/optim.py`, add an argument `weight_decay=0.0` ($\lambda$) to your `SGD`, after `momentum`. Before the momentum update, the gradient of every parameter is `p.grad + weight_decay * p.data`; do not modify `p.grad` itself (same argument as `torch.optim.SGD`).

Test it with `python -m pytest tests/test_optim.py -k weight_decay`.

In [ ]:
run_and_record("weight decay", "wide", weight_decay=1e-2)
compare_tool("weight decay", ["wide"])

### Exercise 1.3: Early stopping

Early stopping stops training when the network starts to memorize: the validation loss is the signal, and `patience` makes the rule robust to noisy curves. It keeps the state of the network at the epoch with the lowest validation loss, and stops once the validation loss has not improved for `patience` consecutive epochs. `fit` calls `step` after every epoch and `restore` at the end.

The state is made of the parameters and of the **buffers**: arrays that belong to a module but are not learned, such as the running statistics of the batch normalization layer of Exercise 2.3 (`model.parameters()` and `model.buffers()` list them). Complete the blanks (a notebook cell, no `minitorch` code).

In [ ]:
class EarlyStopping:
    def __init__(self, patience):
        self.patience = patience
        self.best_loss = np.inf
        self.best_state = None
        self.epochs_without_improvement = 0

    def step(self, val_loss, model):
        """Called after every epoch with the validation loss. Returns True when training must stop."""
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_state = ...                        # a copy of the values of every parameter and every buffer
            self.epochs_without_improvement = 0
        else:
            self.epochs_without_improvement = ...
        return ...                                       # True once `patience` epochs went by without improvement

    def restore(self, model):
        """Put the best state back in the model."""
        for a, saved in zip(model.parameters() + model.buffers(), self.best_state):
            ...

In [ ]:
from tests.lab07_checks import check_early_stopping
check_early_stopping(EarlyStopping)

In [ ]:
run_and_record("early stopping", "wide", patience=20)
compare_tool("early stopping", ["wide"])

for tool in ("baseline", "random crop", "weight decay", "early stopping"):
    show_results("wide", [tool])

**Question 1.2.** How does each tool change the training and validation curves? Which one works best here, and why?

*Answer:*

**Question 1.3.** Would a horizontal flip be a good augmentation for letters? And a 180° rotation? What must hold for a transformation to be a valid augmentation?

*Answer:*

## Part 2: Optimization failure

The **deep** network has 10 hidden layers of 64 units. Trained from small random weights, it does not learn anything. Unlike in Part 1, even the training curves are flat.

In [ ]:
run_and_record("baseline", "deep")
plot_runs(["baseline"], "deep")

### Where does the gradient go?

`layer_grad_norms` measures the gradient that reaches each layer. Compare a network with sigmoid activations and one with ReLU activations (all weights are scaled by 0.1, because the default standard normal weights saturate every sigmoid). Run the cell below with sigmoid, then set `activation = ReLU` and run it again: both profiles appear on the same plot.

In [ ]:
loss_fn = CrossEntropyLoss()
xb, yb = next(iter(make_loaders(1024, 128)[0]))
profiles = {}                                     # gradient norms of every network tried below

In [ ]:
activation = Sigmoid                              # run once, then set it to ReLU and run the cell again
np.random.seed(0)
model = make_mlp(depth=10, width=64, activation=activation)
for p in model.parameters():
    if p.ndim == 2:
        p.data *= 0.1                             # weights with standard deviation 0.1 instead of 1
profiles[activation.__name__] = layer_grad_norms(model, loss_fn, xb, yb)

for name, norms in profiles.items():
    plt.semilogy(norms, marker="o", label=name)
plt.xlabel("layer (0 = first)")
plt.ylabel("gradient norm at the layer's input")
plt.legend()
plt.show()

### Initialization scale

With ReLU, the gradient no longer saturates, yet it still shrinks with depth: the scale of the weights matters too. To study it, we need to choose it.

#### Exercise 2.1: `normal_`

Implement `normal_(param, mean=0.0, std=1.0)` in `minitorch/init.py`: it overwrites `param.data` in place with samples from $\mathcal{N}(\text{mean}, \text{std}^2)$ and returns nothing (same signature as `torch.nn.init.normal_`).

Test it with `python -m pytest tests/test_init.py -k normal`.

Then train the deep network from $\mathcal{N}(0, \sigma^2)$ weights for three values of $\sigma$, and compare the gradient profiles and the training curves.

In [ ]:
stds = (0.01, 0.1, 1.0)
for std in stds:
    run_and_record(f"std {std}", "deep", init=std)

for std in stds:
    plt.semilogy(results[(f"std {std}", "deep")]["grad_norms"], marker="o", label=f"std={std}")
plt.xlabel("layer (0 = first)")
plt.ylabel("gradient norm at the layer's input")
plt.legend()
plt.show()

plot_runs([f"std {std}" for std in stds], "deep")

#### Exercise 2.2: He initialization

All the hidden layers have the same width, so a single $\sigma$ can suit all of them: He initialization derives it from the number of inputs of the layer.

Implement `kaiming_normal_(param)` in `minitorch/init.py`: He initialization for ReLU networks, which fills `param.data` in place with samples from $\mathcal{N}(0, 2 / \text{fan\_in})$, where `fan_in` is the number of inputs of the layer (`param.shape[1]` for a weight of shape `(out_features, in_features)`). Same name as `torch.nn.init.kaiming_normal_` with `nonlinearity="relu"`.

Test it with `python -m pytest tests/test_init.py -k kaiming`.

In [ ]:
run_and_record("He init", "deep", init="he")

for name in ("std 0.1", "He init"):
    plt.semilogy(results[(name, "deep")]["grad_norms"], marker="o", label=name)
plt.xlabel("layer (0 = first)")
plt.ylabel("gradient norm at the layer's input")
plt.legend()
plt.show()

compare_tool("He init", ["deep"])

### Normalization and skip connections

He initialization only sets the scale at the start of training. Two architectural tools keep the network well-behaved throughout: normalization and skip connections.

#### Exercise 2.3: Batch normalization

Implement the forward pass of `BatchNorm1d(num_features, eps=1e-5, momentum=0.1)` in `minitorch/layer.py`. The backward pass is provided.

**This layer behaves differently in training and in eval mode.** `self.training` tells which mode applies (`model.train()` and `model.eval()` set it, see Part 0.2). Your `forward` needs both branches:

| Mode | Statistics used to normalize | Running statistics |
|---|---|---|
| training | mean and variance of the batch | updated |
| eval | running mean and variance | unchanged |

In eval mode, the output for an image must not depend on the other images of the batch: at test time there may be a single image, and the running statistics, which estimate the statistics of the training data, replace the batch statistics.

In training mode, with $\mathbf{x}_i$ the $N$ rows of the batch (all operations are elementwise over the features):

$$\boldsymbol{\mu} = \frac{1}{N}\sum_i \mathbf{x}_i, \qquad \boldsymbol{\sigma}^2 = \frac{1}{N}\sum_i (\mathbf{x}_i - \boldsymbol{\mu})^2, \qquad \hat{\mathbf{x}}_i = \frac{\mathbf{x}_i - \boldsymbol{\mu}}{\sqrt{\boldsymbol{\sigma}^2 + \epsilon}}, \qquad \mathbf{y}_i = \boldsymbol{\gamma} \odot \hat{\mathbf{x}}_i + \boldsymbol{\beta}.$$

- `self.weight` ($\boldsymbol{\gamma}$) and `self.bias` ($\boldsymbol{\beta}$) are `Parameter` objects, already defined.
- `self.running_mean` and `self.running_var` are `Buffer`s: arrays of the state of the layer that are not learned. Update them **in place** through `.data`: $\mathbf{r} \leftarrow (1 - m)\,\mathbf{r} + m\,\mathbf{s}$, where $m$ is `momentum` and $\mathbf{s}$ is the mean (resp. variance) of the batch. This `momentum` plays a role similar to the optimizer's: both are moving averages that smooth a noisy signal, here the batch statistics. It is nonetheless a different hyperparameter, with the opposite convention: $m$ weighs the *new* batch, whereas the momentum of SGD weighs the past.
- `forward` must save the pair `(x_hat, inv_std)` in `self.cache` for the provided `backward`, where `inv_std` is $1 / \sqrt{\boldsymbol{\sigma}^2 + \epsilon}$.

Test it with `python -m pytest tests/test_layer.py -k BatchNorm`.

#### Exercise 2.4: Residual connection

Implement `Residual(module)` in `minitorch/layer.py`: it wraps a skip connection around another module with equal input and output shapes, $\mathbf{y} = \mathbf{x} + f(\mathbf{x})$, where $f$ is `module`. Implement `forward`, `backward` and `parameters`.

Test it with `python -m pytest tests/test_layer.py -k Residual`.

In [ ]:
run_and_record("batch norm", "deep", norm=True)
compare_tool("batch norm", ["deep"])

run_and_record("residual", "deep", residual=True)
compare_tool("residual", ["deep"])

for tool in ("baseline", "He init", "batch norm", "residual"):
    show_results("deep", [tool])

**Question 2.1.** Would increasing the learning rate be a viable solution to fix vanishing gradients? Why or why not? (You can try it: `run_and_record("learning rate", "deep", lr=...)`.)

*Answer:*

## Part 3: Stretch goal

Each tool was introduced for one network. Do they also help, or hurt, on the other one? Test every tool on both networks. Before running anything, write your predictions in the table below (*helps*, *hurts* or *no effect*).

| Tool | wide: overfitting | deep: does not train |
|---|---|---|
| He init | | |
| Batch normalization | | |
| Residual connections | | |
| Weight decay | | |
| Random crops | | |
| Early stopping | | |

In [ ]:
TOOLS = {
    "He init": dict(init="he"),
    "batch norm": dict(norm=True),
    "residual": dict(residual=True),
    "weight decay": dict(weight_decay=1e-2),
    "random crop": dict(crop_padding=2),
    "early stopping": dict(patience=20),
}

for tool, changes in TOOLS.items():
    for net in ("wide", "deep"):
        if (tool, net) not in results:            # skip the experiments of Parts 1 and 2
            run_and_record(tool, net, **changes)
    compare_tool(tool)

for net in ("wide", "deep"):
    show_results(net, ["baseline", *TOOLS])
    print()

### Exercise 3.1

Find a combination of tools that makes the deep network both train and generalize as well as you can.

In [ ]:
run_and_record("combination", "deep")     # add the tools you choose as keyword arguments

---

To run all the tests of the lab: `python -m pytest tests`